# 07 · Auditoría de unidades y diseño del siguiente banco
**Exclusivamente en Colab. Plan prospectivo; no genera ni acepta un banco nuevo.**

El banco actual conserva sus unidades y evaluador. Esta auditoría separa un problema
dimensional de una calibración empírica: multiplicar densidades por 60 no establece
que el tráfico resultante sea realista. No ajusta ruido para favorecer a un método.

El inventario agrupa las filas del catálogo por geometría antes de asignarlas a
entrenamiento/validación/test. Excluye las cuatro geometrías ya vistas. Si el catálogo
no alcanza las cuotas, lo informa: más semillas o ventanas no llenan ese faltante.
Las cuotas son objetivos logísticos, **no un cálculo de potencia**. Hace falta auditar
la independencia de las fuentes y construir/aceptar el nuevo banco en Colab antes de
entrenar con él. Este notebook no habilita p-valores sobre el piloto.


In [1]:
import sys, os, json, shutil, subprocess, importlib.metadata
from pathlib import Path
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('Por instrucción del autor: ejecutar exclusivamente en Google Colab.') from exc
BANK_REFERENCE = {'bank_id': 'pilot00_280553a33c2ab594', 'protocol_version': 'pilot-00-v4', 'manifest_sha256': 'd8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02', 'acceptance_status': 'ready_for_method_development', 'role': 'development_only', 'validation_run_id': '20260917T021417Z_665a3a34', 'evidence': 'docs/NOTEBOOK_00_VALIDATION.md'}
BANK_ID = BANK_REFERENCE['bank_id']
SUPPORTED_BANK_PROTOCOLS = (BANK_REFERENCE['protocol_version'],)
RESUME_RUN_ID = ''
ENABLE_GUROBI = False
WORK = Path('/content/TESIS_DESIGN_07')
RUNTIME = WORK / 'runtime'
(RUNTIME / 'svrplab').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'tests').mkdir(exist_ok=True)
(RUNTIME / 'svrplab/__init__.py').write_text('"""Métodos y evaluador sellado del banco de tesis."""\n')
os.chdir(RUNTIME)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
needed = ['numpy', 'pandas', 'matplotlib', 'pytest', 'ortools']
if ENABLE_GUROBI:
    needed.append('gurobipy')
missing = [name for name in needed if version(name) is None]
if missing:
    constraints = RUNTIME / 'constraints.txt'
    protected = ['numpy','pandas','matplotlib','scipy','protobuf','torch','pillow']
    constraints.write_text(''.join(f'{name}=={version(name)}\n' for name in protected if version(name)))
    requirements = ['gurobipy>=11,<14' if name == 'gurobipy' else ('ortools>=9.11,<10' if name == 'ortools' else name) for name in missing]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), *requirements], check=True)
print('Python:', sys.version)
print('Dependencias:', {name: version(name) for name in needed})
print('CPU; no se solicita GPU para estos métodos.')


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Dependencias: {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.0', 'pytest': '8.4.2', 'ortools': '9.12.4544'}
CPU; no se solicita GPU para estos métodos.


In [2]:
DESIGN = {'schema_version': 'research-design-v1', 'status': 'prospective_plan_not_accepted_bank', 'sizes': [50, 100, 200, 300], 'split_seed': 20260917, 'geometry_quotas_per_size': {'train': 100, 'validation': 20, 'test': 30}, 'fr_absolute_error': 0.05, 'fr_error_probability': 0.05, 'cvar_alpha': 0.95, 'minimum_expected_tail_observations': 50, 'primary_endpoint': 'paired_geometry_mean_pipeline_strict_feasibility', 'secondary_endpoints': ['conditional_travel_on_common_feasible_cohort', 'cvar_travel', 'total_compute'], 'power_guaranteed': False, 'quotas_are_planning_targets_not_a_power_calculation': True, 'test_exposure_policy': 'once_after_config_and_models_frozen', 'failed_runs_policy': 'preserve_all_failures_missing_are_not_solver_failures', 'unit_model_change_policy': 'separate_version_never_edit_accepted_bank', 'objective_scope': 'conservative_travel_with_risk_evaluated_separately_no_claim_c_plus_Q'}


In [3]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/TESIS-EHBG-FACS')
import hashlib
def verify_before_import(folder):
    folder = Path(folder)
    manifest = json.loads((folder / 'CHECKSUMS.json').read_text())
    actual = {str(f.relative_to(folder)) for f in folder.rglob('*')
              if f.is_file() and f.name != 'CHECKSUMS.json' and '__pycache__' not in f.relative_to(folder).parts}
    if actual != set(manifest):
        raise ValueError('Inventario del banco distinto del manifiesto.')
    for relative, expected in manifest.items():
        file = folder / relative
        if not file.resolve().is_relative_to(folder.resolve()):
            raise ValueError('Ruta de manifiesto fuera del banco.')
        if hashlib.sha256(file.read_bytes()).hexdigest() != expected:
            raise ValueError(f'Hash incorrecto: {relative}')
    return manifest
if BANK_ID != BANK_REFERENCE['bank_id']:
    raise ValueError('BANK_ID difiere de la referencia validada; actualizar la referencia común antes de cambiar banco.')
remote_bank = DRIVE_ROOT / 'banks' / BANK_ID
manifest = verify_before_import(remote_bank)
manifest_sha = hashlib.sha256(json.dumps(manifest, sort_keys=True, separators=(',', ':'),
                                        ensure_ascii=False, allow_nan=False).encode('utf-8')).hexdigest()
if manifest_sha != BANK_REFERENCE['manifest_sha256']:
    raise ValueError('El banco no coincide con el manifiesto aceptado en 00; no importar ni ejecutar.')
acceptance = json.loads((remote_bank / 'acceptance.json').read_text())
if (acceptance.get('status') != BANK_REFERENCE['acceptance_status']
        or acceptance.get('protocol_version') != BANK_REFERENCE['protocol_version']):
    raise ValueError('El acta no coincide con la aceptación esperada del banco.')
BANK = WORK / 'banks' / BANK_ID
if not BANK.exists():
    BANK.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(remote_bank, BANK)
if verify_before_import(BANK) != manifest:
    raise ValueError('La copia local no coincide con Drive.')
shutil.copy2(BANK / 'code/svrplab_pilot.py', RUNTIME / 'svrplab/pilot.py')
print('Banco validado en 00, copiado y hashes verificados:', BANK_ID, manifest_sha)


Mounted at /content/drive
Banco validado en 00, copiado y hashes verificados: pilot00_280553a33c2ab594 d8fe8d222f14acb9a32acfc62cdb105f3f364b85d47a4e8c96c7ef58c426fc02


In [4]:
%%writefile svrplab/research_design.py
"""Diseño previo y auditoría de unidades. Ejecutar cálculos sólo en Colab.

No cambia el evaluador sellado, no genera instancias ni certifica independencia
estadística a partir de hashes. No elige datos por el desempeño de métodos.
"""
from __future__ import annotations

from collections import defaultdict
import hashlib
import json
import math


def _hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     ensure_ascii=False, allow_nan=False).encode()).hexdigest()


def plan_geometry_split(records, excluded_groups, config):
    """Plan reproducible por geometría y tamaño, anterior a construir ventanas.

    records: geometry_group_sha256, n_customers, source_id. Todos los derivados
    comparten rol. Escasez no se solventa replicando geometrías ni bajando cuotas.
    """
    if config['schema_version'] != 'research-design-v1':
        raise ValueError('Diseño desconocido.')
    quotas = config['geometry_quotas_per_size']
    if set(quotas) != {'train', 'validation', 'test'} or any(type(v) is not int or v < 1 for v in quotas.values()):
        raise ValueError('Cuotas por geometría inválidas.')
    excluded = set(excluded_groups)
    groups, sizes = defaultdict(list), {}
    for row in records:
        group = row['geometry_group_sha256']
        if len(group) != 64 or any(c not in '0123456789abcdef' for c in group):
            raise ValueError('Hash de geometría inválido.')
        n = row['n_customers']
        if type(n) is not int or n <= 0:
            raise ValueError('Tamaño inválido.')
        if group in sizes and sizes[group] != n:
            raise ValueError('Una geometría aparece con tamaños inconsistentes.')
        sizes[group] = n
        groups[group].append(row['source_id'])
    planned, shortages = [], []
    for n in config['sizes']:
        available = sorted((g for g in groups if sizes[g] == n and g not in excluded),
                           key=lambda g: (_hash([config['split_seed'], g]), g))
        offset = 0
        for role in ('train', 'validation', 'test'):
            selected = available[offset:offset + quotas[role]]
            if len(selected) < quotas[role]:
                shortages.append({'n_customers': n, 'role': role, 'required': quotas[role],
                                  'available': len(selected)})
            for group in selected:
                planned.append({'geometry_group_sha256': group, 'n_customers': n,
                                'role': role, 'source_ids': sorted(set(groups[group]))})
            offset += quotas[role]
    return {'schema_version': config['schema_version'], 'config': config,
            'assignment': planned, 'excluded_development_groups': sorted(excluded),
            'shortages': shortages, 'status': 'insufficient_population' if shortages else 'planned_not_validated',
            'ready_for_confirmatory': False,
            'independence_requires_source_audit': True,
            'plan_sha256': _hash({'config': config, 'assignment': planned, 'excluded': sorted(excluded)})}


def scenario_precision_plan(config):
    """Hoeffding por plan para FR; tamaño de cola es diagnóstico, no precisión CVaR."""
    epsilon, error = config['fr_absolute_error'], config['fr_error_probability']
    alpha, tail = config['cvar_alpha'], config['minimum_expected_tail_observations']
    if not 0 < epsilon < 1 or not 0 < error < 1 or not 0 < alpha < 1 or type(tail) is not int or tail < 1:
        raise ValueError('Precisión/cola fuera del dominio.')
    fr_n = math.ceil(math.log(2. / error) / (2. * epsilon**2))
    tail_n = math.ceil(tail / (1. - alpha))
    return {'scenarios_per_plan': max(fr_n, tail_n), 'fr_hoeffding_count': fr_n,
            'cvar_tail_count_floor': tail_n,
            'fr_scope': 'pointwise_per_fixed_plan_iid_scenarios_not_simultaneous_all_methods',
            'cvar_precision_guaranteed': False,
            'scenario_count_is_not_number_of_independent_geometries': True,
            'note': 'Congelar M antes de abrir test. No detener al obtener significancia.'}


def traffic_density_audit():
    """Identidad de unidades, NO recomendación de multiplicar el ruido por 60."""
    def pdf(t, mu, sigma):
        return math.exp(-.5 * ((t - mu) / sigma)**2) / (sigma * math.sqrt(2. * math.pi))
    rows = []
    for t in (0, 360, 480, 780, 1020, 1260, 1440):
        minute = pdf(t, 480, 90) + pdf(t, 1020, 90)
        hour = pdf(t / 60., 8, 1.5) + pdf(t / 60., 17, 1.5)
        rows.append({'time_minutes': t, 'density_per_minute': minute,
                     'density_per_hour': hour, 'hour_to_minute_density_ratio': hour / minute})
    return {'status': 'dimensional_discrepancy_requires_versioned_model_decision', 'rows': rows,
            'sealed_evaluator_modified': False,
            'reason': 'La densidad tiene unidades inversas al tiempo; amplitudes y tasa por intervalo también necesitan unidades. '
                      'Cambiar sólo sigma/t aumenta amplitudes y NO es una conversión física completa.',
            'paper': 'https://arxiv.org/html/2505.21887v1#S2.SS1'}


Writing svrplab/research_design.py


In [5]:
%%writefile tests/test_research_design.py
"""Pruebas de diseño; ejecutar sólo en Colab como las demás pruebas del proyecto."""
import copy
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import research_design as r


def config():
    return {'schema_version': 'research-design-v1', 'sizes': [50], 'split_seed': 7,
            'geometry_quotas_per_size': {'train': 1, 'validation': 1, 'test': 1}}


def test_splits_are_grouped_exclude_development_and_order_invariant():
    rows = [{'geometry_group_sha256': f'{i:064x}', 'n_customers': 50, 'source_id': str(i)} for i in range(5)]
    rows.append(dict(rows[1], source_id='derivative'))
    excluded = [rows[0]['geometry_group_sha256']]
    first = r.plan_geometry_split(rows, excluded, config())
    second = r.plan_geometry_split(list(reversed(rows)), excluded, config())
    assert first == second
    assert not first['shortages'] and not first['ready_for_confirmatory']
    assert not set(excluded) & {row['geometry_group_sha256'] for row in first['assignment']}
    assert len({row['geometry_group_sha256'] for row in first['assignment']}) == 3


def test_replicas_cannot_fill_geometry_quotas():
    row = {'geometry_group_sha256': 'a' * 64, 'n_customers': 50, 'source_id': 'one'}
    result = r.plan_geometry_split([copy.deepcopy(row) for _ in range(100)], [], config())
    assert result['status'] == 'insufficient_population' and len(result['assignment']) == 1


def test_density_audit_and_precision_do_not_authorize_new_model_or_test():
    result = r.traffic_density_audit()
    assert not result['sealed_evaluator_modified']
    assert all(abs(row['hour_to_minute_density_ratio'] - 60.) < 1e-8 for row in result['rows'])
    precision = r.scenario_precision_plan({'fr_absolute_error': .05, 'fr_error_probability': .05,
                                         'cvar_alpha': .95, 'minimum_expected_tail_observations': 50})
    assert precision['scenarios_per_plan'] >= 1000
    assert not precision['cvar_precision_guaranteed']


Writing tests/test_research_design.py


In [6]:
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
import importlib
for name in list(sys.modules):
    if name == 'svrplab' or name.startswith('svrplab.'):
        del sys.modules[name]
importlib.invalidate_caches()
from svrplab import pilot as p
from svrplab import research_design as rd
import pandas as pd
from IPython.display import display
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_research_design.py'], check=True)
instances = p.load_bank(BANK)
UNITS = rd.traffic_density_audit()
PRECISION = rd.scenario_precision_plan(DESIGN)
display(pd.DataFrame(UNITS['rows']))
print(UNITS['reason'])
print(json.dumps(PRECISION, indent=2))


,time_minutes,density_per_minute,density_per_hour,hour_to_minute_density_ratio
0,0,2.951447e-09,1.770868e-07,60.0
1,360,1.822334e-03,1.093400e-01,60.0
2,480,4.432692e-03,2.659615e-01,60.0
3,780,1.437585e-04,8.625510e-03,60.0
4,1020,4.432692e-03,2.659615e-01,60.0
5,1260,1.266221e-04,7.597324e-03,60.0
6,1440,8.273384e-08,4.964031e-06,60.0


La densidad tiene unidades inversas al tiempo; amplitudes y tasa por intervalo también necesitan unidades. Cambiar sólo sigma/t aumenta amplitudes y NO es una conversión física completa.
{
  "scenarios_per_plan": 1000,
  "fr_hoeffding_count": 738,
  "cvar_tail_count_floor": 1000,
  "fr_scope": "pointwise_per_fixed_plan_iid_scenarios_not_simultaneous_all_methods",
  "cvar_precision_guaranteed": false,
  "scenario_count_is_not_number_of_independent_geometries": true,
  "note": "Congelar M antes de abrir test. No detener al obtener significancia."
}


La cota Hoeffding controla el error de FR de **un plan fijo** bajo escenarios iid;
no es un intervalo simultáneo sobre todos los métodos ni convierte escenarios en
geometrías independientes. El número esperado de observaciones de cola no garantiza
precisión de CVaR. El M actual sellado permanece intacto.

Se usa el catálogo oficial fijado por hash. Las ventanas repetidas de una geometría
quedan juntas. Un hash diferente no prueba independencia de origen: esa auditoría
sigue siendo necesaria antes de un test confirmatorio.


In [7]:
rows, source = p.load_catalog(WORK / 'catalog_cache')
records = []
for row in rows:
    subset = row['subset_name']
    if not subset.startswith('twvrp_') or not subset.endswith('_single_depot'):
        continue
    n = len(row['locations']) - 1
    if n not in DESIGN['sizes']:
        continue
    records.append({'geometry_group_sha256': p.digest({'locations': [[float(value) for value in coordinate] for coordinate in row['locations']], 'depot_indices': [0]}),
                    'n_customers': n, 'source_id': subset + ':' + str(row['instance_id'])})
excluded = [x.provenance['geometry_group_sha256'] for x in instances]
PLAN = rd.plan_geometry_split(records, excluded, DESIGN)
display(pd.DataFrame(PLAN['assignment']))
display(pd.DataFrame(PLAN['shortages']))
print('Estado:', PLAN['status'], '| No habilita inferencia confirmatoria.')
from datetime import datetime, timezone
import uuid
OUT = WORK / 'research_design' / (PLAN['plan_sha256'][:12] + '_' + uuid.uuid4().hex[:8])
OUT.mkdir(parents=True, exist_ok=True)
p.write_json(OUT / 'plan.json', PLAN)
p.write_json(OUT / 'units_audit.json', UNITS)
p.write_json(OUT / 'precision_plan.json', PRECISION)
p.write_json(OUT / 'source.json', source)
p.write_json(OUT / 'environment.json', p.environment_report())
p.seal_bundle(OUT)
p.sync_verified(OUT, DRIVE_ROOT / 'research_design' / OUT.name)
print('Diseño guardado:', DRIVE_ROOT / 'research_design' / OUT.name)


data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  495kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

,geometry_group_sha256,n_customers,role,source_ids
0,dd64c554742d4aff6ba817842c704382ce35e5cb8e6964...,50,train,"[twvrp_50_single_depot:5, twvrp_50_single_depo..."
1,8aadcb2ab613a35ff88a461f30aad75fed540db5a24696...,100,train,"[twvrp_100_single_depot:5, twvrp_100_single_de..."
2,c4415794a446747229ede5395041ca56afe574418a11d5...,200,train,"[twvrp_200_single_depot:5, twvrp_200_single_de..."


,n_customers,role,required,available
0,50,train,100,1
1,50,validation,20,0
2,50,test,30,0
3,100,train,100,1
4,100,validation,20,0
5,100,test,30,0
6,200,train,100,1
7,200,validation,20,0
8,200,test,30,0
9,300,train,100,0


Estado: insufficient_population | No habilita inferencia confirmatoria.
Diseño guardado: /content/drive/MyDrive/TESIS-EHBG-FACS/research_design/905a04b22140_54565059
